# CurvLearn — Dimension sweep: does curvature help once space is tight?

The mechanism comparison (gyro) halved the hyperbolic penalty on the hard task at d_model=32
but didn't cross below flat. Hypothesis (and the standard hyperbolic-embedding result, and
prior experience): **hyperbolic's advantage is a low-dimension phenomenon** — give the model
enough Euclidean dimensions and it packs the hierarchy flat, so curvature stops mattering.

This sweeps **d_model ∈ {4, 8, 16, 32}** for the gyro mechanism on medium+hard Dyck, fixed κ,
over seeds. If the thesis holds, hyperbolic aggregation should **beat flat at small d_model**,
first and most on the hard task.

GPU runtime. Results persist to Drive; checkpointed/resumable.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'CPU (slow)'

## 2. Code + install (after a `git pull`, restart the runtime to pick up changes)


In [ ]:
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Drive + output


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 4. Run the dimension sweep
4 dims x 4 κ x 2 difficulties x 3 seeds = 96 runs (small dims train fast), ~45-60 min on an
A100. Checkpointed to dim_sweep_results.json. Drop `seeds` to 2 for a quicker first look.


In [ ]:
import json
from curvlearn.dim_sweep import run_dim_sweep, default_spec

spec = default_spec()
spec['train']['device'] = dev
ckpt = os.path.join(OUTDIR, 'dim_sweep_results.json')
out = run_dim_sweep(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 5. Plot: hyperbolic advantage vs dimension


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'dim_sweep_results.json')))
png = os.path.join(OUTDIR, 'dim_sweep.png')
plot.dim_sweep_plot(out, png)
display(Image(png))
print('saved', png)

## 6. Read it
- **Left panel dips below 0 at small d_model (hard first)** → the crossover exists: hyperbolic
  aggregation beats flat once the dimension is a binding constraint. This is the regime where
  tunable curvature helps a sequence model — Design B (scheduling κ) now has a home.
- **Stays above 0 at every dim** → even at d_model=4 the single hyperbolic aggregation term is
  too diluted by the Euclidean residual+FFN; next is genuinely hyperbolic layers (Möbius
  linear, hyperbolic FFN) that keep the representation on the manifold throughout.
- Watch the right panel (curves at the smallest dim): a clear dip at κ<0 for hard is the
  positive signature.
